## Krok 1 — Načítanie a oprava zoznamu porúch

Načíta súbor `Zoznam_evidovaných_poruch_merania.xlsx` so zoznamom evidovaných porúch merania.
Opravuje zlúčené bunky v Exceli — pandas načíta prázdne bunky ako NaN, preto sa aplikuje `ffill()` na stĺpec EIC kód.
Kontroluje EIC s viacerými typmi porúch.

> Úplný zoznam vstupných dát je dostupný v prílohe bakalárskej práce (ZIP súbor).

In [ ]:
import pandas as pd
import numpy as np


# Cesta k vstupnému súboru
input_file = os.path.join("data", "Zoznam_evidovaných_poruch_merania.xlsx")

# Načítanie súboru s typmi porúch
faults = pd.read_excel(input_file)


# Oprava zlúčených buniek — pandas načíta prázdne bunky ako NaN
# po rozdelení zlúčených buniek v Exceli
faults['EIC kód'] = faults['EIC kód'].ffill()

# Kontrola po oprave
print("Po oprave:")
print(faults[['EIC kód', 'Typ poruchy', 'data_od', 'data_do']].to_string(index=False))
print(f"\nPočet unikátnych EIC: {faults['EIC kód'].nunique()}")
print(f"Celkový počet riadkov: {len(faults)}")

# Kontrola EIC s viacerými typmi porúch
grouped = (
    faults
    .groupby('EIC kód')['Typ poruchy']
    .apply(list)
    .reset_index()
)
multi = grouped[grouped['Typ poruchy'].apply(len) > 1]
print(f"\nEIC s viacerými typmi porúch: {len(multi)}")
for _, row in multi.iterrows():
    print(f"  EIC: {row['EIC kód']} | Typy: {row['Typ poruchy']}")
    
# Uloženie opraveného datasetu do priečinka data
output_file = os.path.join("data", "typy_fixed.xlsx")

faults.to_excel(output_file, index=False)

print("\nUložené: typy_fixed.xlsx")

## Krok 2 — Označenie záznamov s poruchou

Načíta surové merania zo súboru `ims_tuke_poruchy_28072025.parquet` a zoznam porúch z `typy_fixed.xlsx`.
Pre každý záznam určí či sa nachádza v intervale poruchy (data_od — data_do) pre daný EIC.
Záznamy v intervale poruchy dostanú hodnotu `porucha?=1`, ostatné `porucha?=0`.
Ak chýba dátum ukončenia poruchy — použije sa dnešný dátum.
> Všetky vstupné a výstupné súbory sú dostupné v prílohe bakalárskej práce (ZIP súbor).



In [ ]:
import pandas as pd
import numpy as np
import os

# =========================
# Načítanie dát
# =========================

energy_file = os.path.join("data", "ims_tuke_poruchy_28072025.parquet")
faults_file = os.path.join("data", "typy_fixed.xlsx")

df_energy = pd.read_parquet(energy_file)

df_energy['t_utc'] = pd.to_datetime(
    df_energy['t_utc'],
    errors='coerce'
).dt.tz_localize(None)

df_energy = df_energy.dropna().reset_index(drop=True)

# =========================
# Načítanie intervalov porúch
# =========================

faults = pd.read_excel(faults_file)

faults['data_od'] = pd.to_datetime(faults['data_od'])
faults['data_do'] = pd.to_datetime(faults['data_do'])

# Ak chýba dátum ukončenia poruchy
faults['data_do'] = faults['data_do'].fillna(pd.Timestamp("today"))

# =========================
# Označenie záznamov s poruchou
# =========================

df_energy['porucha?'] = 0
df_energy['_orig_idx'] = df_energy.index

energy_small = df_energy[['_orig_idx', 'eic', 't_utc']].copy()

faults_ranges = (
    faults[['EIC kód', 'data_od', 'data_do']]
    .drop_duplicates()
    .rename(columns={'EIC kód': 'eic'})
)

merged = pd.merge(
    energy_small,
    faults_ranges,
    on='eic',
    how='inner'
)

condition = (
    (merged['t_utc'] >= merged['data_od']) &
    (merged['t_utc'] <= merged['data_do'])
)

fault_indices = merged.loc[condition, '_orig_idx'].unique()

df_energy.loc[
    df_energy['_orig_idx'].isin(fault_indices),
    'porucha?'
] = 1

df_energy.drop(columns=['_orig_idx'], inplace=True)

# =========================
# Výstup
# =========================

print(f"Záznamy s poruchou (porucha=1): {df_energy['porucha?'].sum()}")
print(f"Záznamy bez poruchy (porucha=0): {(df_energy['porucha?'] == 0).sum()}")

# Uloženie výstupu do priečinka data
output_file = os.path.join("data", "energy_with_faults_typy.csv")

df_energy.to_csv(output_file, index=False)

print(f"Uložené: {output_file}")

## Krok 3 — Agregácia časových radov do 6-hodinových blokov

Načíta dataset s označenými poruchami `energy_with_faults_typy.csv` a agreguje surové merania signálov napätia (u1, u2, u3) a prúdu (i1, i2, i3) do 6-hodinových blokov.

Pre každý signál sa vypočítavajú nasledujúce štatistické charakteristiky:
- min, max, mean, std
- peak-to-peak (p2p)
- skewness, kurtosis
- počet núl (zeros_count)
- počet výkyvov (spikes_count)
- priemerná absolútna zmena (mean_abs_diff)

Cieľová premenná `porucha?` sa agreguje funkciou `max` — blok je označený ako poruchový ak obsahuje aspoň jeden poruchový záznam.


> Všetky vstupné a výstupné súbory sú dostupné v prílohe bakalárskej práce (ZIP súbor).

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay
)
import matplotlib.pyplot as plt

# =========================
# Načítanie dát
# =========================

input_file = os.path.join("data", "energy_with_faults_typy.csv")

df_new = pd.read_csv(input_file)

df_new["t_utc"] = pd.to_datetime(df_new["t_utc"])

signal_cols = [
    "u1_norm", "u2_norm", "u3_norm",
    "i1_norm", "i2_norm", "i3_norm"
]

# Pomocné funkcie pre agregáciu
def mean_abs_diff(x):
    return x.diff().abs().mean()

def count_spikes(x, threshold=3):
    limit = x.mean() + threshold * x.std()
    return (x > limit).sum()

def evaluate_model(y_true, y_pred, labels=None, title="Matica zámen"):
    """
    Výpis klasifikačnej správy a zobrazenie matice zámen.

    Parametre:
        y_true  - skutočné hodnoty
        y_pred  - predikované hodnoty
        labels  - názvy tried
        title   - názov grafu
    """
    print("=" * 60)
    print(f"Klasifikačná správa — {title}")
    print("=" * 60)
    print(classification_report(y_true, y_pred, target_names=labels))

    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=labels
    )
    fig, ax = plt.subplots(figsize=(6, 5))
    disp.plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(title)
    plt.tight_layout()
    plt.show()

    return cm

# Definícia agregačných funkcií pre každý signál
agg_dict = {}
for col in signal_cols:
    agg_dict[f"{col}_min"]           = (col, "min")
    agg_dict[f"{col}_max"]           = (col, "max")
    agg_dict[f"{col}_mean"]          = (col, "mean")
    agg_dict[f"{col}_std"]           = (col, "std")
    agg_dict[f"{col}_p2p"]           = (col, lambda x: x.max() - x.min())
    agg_dict[f"{col}_skew"]          = (col, lambda x: x.skew())
    agg_dict[f"{col}_kurtosis"]      = (col, lambda x: x.kurtosis())
    agg_dict[f"{col}_zeros_count"]   = (col, lambda x: (x == 0).sum())
    agg_dict[f"{col}_spikes_count"]  = (col, count_spikes)
    agg_dict[f"{col}_mean_abs_diff"] = (col, mean_abs_diff)

agg_dict["porucha?"] = ("porucha?", "max")

# Agregácia časových radov po 6-hodinových blokoch
daily_df_new = (
    df_new
    .groupby([
        "eic",
        pd.Grouper(key="t_utc", freq="6h")
    ])
    .agg(**agg_dict)
    .reset_index()
)

daily_df_new["date"]       = daily_df_new["t_utc"].dt.floor("D")
daily_df_new["time_block"] = daily_df_new["t_utc"].dt.hour // 6

# Výstup
print(f"Celkový počet blokov: {len(daily_df_new)}")
print(f"Záznamy s poruchou (porucha=1): {daily_df_new['porucha?'].sum()}")
print(f"Záznamy bez poruchy (porucha=0): {(daily_df_new['porucha?'] == 0).sum()}")

output_file = os.path.join("data", "agg_6h_typy2.csv")

daily_df_new.to_csv(output_file, index=False)


print("Uložené: agg_6h_typy2.csv")

## Krok 4 — Priradenie typov porúch (multi-hot kódovanie)

Načíta agregovaný dataset `agg_6h_typy2.csv` a priradí každému poruchovému bloku konkrétny typ poruchy zo súboru `typy_fixed.xlsx`.

Pre každý typ poruchy sa vytvorí samostatný binárny stĺpec `fault_type_X` (multi-hot kódovanie) — hodnota 1 znamená že daný blok obsahuje poruchu tohto typu.

Priradenie prebieha na základe zhody EIC kódu a časového intervalu (data_od — data_do).

**Výstup:** `data/final_typy.csv` — finálny dataset pripravený pre trénovanie modelov (144 MB)

> Toto je hlavný vstupný súbor pre všetky tri modely. Všetky vstupné a výstupné súbory sú dostupné v prílohe bakalárskej práce (ZIP súbor).

In [ ]:
import pandas as pd
import numpy as np

input_file = os.path.join("data", "agg_6h_typy2.csv")

daily_df = pd.read_csv(input_file)
daily_df["t_utc"] = pd.to_datetime(daily_df["t_utc"])

input_file2 = os.path.join("data", "typy_fixed.xlsx")

faults = pd.read_excel(input_file2)
faults["data_od"] = pd.to_datetime(faults["data_od"])
faults["data_do"] = pd.to_datetime(faults["data_do"])
faults["data_do"] = faults["data_do"].fillna(pd.Timestamp("today"))

# Multi-hot stĺpce
all_fault_types = faults["Typ poruchy"].dropna().unique()
for t in all_fault_types:
    daily_df[f"fault_type_{int(t)}"] = 0
daily_df["Typ poruchy"] = np.nan

# Priradenie podľa t_utc
for _, fault in faults.iterrows():
    mask = (
        (daily_df["eic"] == fault["EIC kód"]) &
        (daily_df["t_utc"] >= fault["data_od"]) &
        (daily_df["t_utc"] <= fault["data_do"]) &
        (daily_df["porucha?"] == 1)
    )
    daily_df.loc[mask, "Typ poruchy"] = fault["Typ poruchy"]
    col = f"fault_type_{int(fault['Typ poruchy'])}"
    if col in daily_df.columns:
        daily_df.loc[mask, col] = 1

# Kontrola
fault_cols = [col for col in daily_df.columns if col.startswith('fault_type_')]
print("NOVÉ POČTY BLOKOV PODĽA TYPOV:")
print("="*50)
for col in sorted(fault_cols, key=lambda x: daily_df[x].sum()):
    count = int(daily_df[col].sum())
    pct   = count / daily_df['porucha?'].sum() * 100
    print(f"  {col:<15} {count:<8} ({pct:.1f}%)")

output_file = os.path.join("data", "final_typy.csv")

daily_df.to_csv(output_file, index=False)

print("\n✅ Uložené: final_typy.csv")